# Calibration

This is the summer4 port of `notebooks/04-Calibration.ipynb` on `tweak_cases_calib`. The density is `vacf.calibration.make_log_likelihood`: the same seven terms, the same error models, and the same uniform priors. Every name in `HOMOGENEOUS_PARAM_BOUNDS` is sampled (mixing parameters are fixed for homogeneous mixing). Everything else in `BASE_PARAMS` stays fixed, which is what that notebook does with `BASE_PARAMS | {sampled}`.

The summer3 local run is `N_RUNS_LOCAL = 40` warmup and 40 samples. This one draws 100 and 100. The kernel matches that notebook: NUTS, tree depth 5, started at `get_nuts_init_values()`, four chains, `PRNGKey(2)`.

The contact matrix is the folded spline, not a kernel rebuilt on every stage. `bg_mixing`, `a_spread` and `pc_strength` stay fixed at their base values. The dynamical model is the corrected default. [Notebook 02](02-ReleaseModel.ipynb) compares that to the golden-matching path. Parameters were originally chosen under the bugs, so the same point is a different epidemic.

Restart the kernel and run from the top. The device count is fixed when JAX is imported. If this kernel has already imported JAX, NumPyro runs the four chains one after another. Use the **vacf (pixi)** kernel, as in [the inspect notebook](01-InspectRun.ipynb).

A depth-4 run of this likelihood took about 9 seconds an iteration, with trees near the cap of 16 leapfrog steps. Depth 5 allows 32, so this draw can take longer than half an hour.

## Four CPU devices

NumPyro's default `chain_method` is `parallel`, and it rewrites that to `sequential` when JAX sees fewer devices than chains. This cell asks for four host devices before JAX is imported, then refuses to continue if that did not happen.


In [ ]:
import os

flags = os.environ.get("XLA_FLAGS", "").split()
kept = [
    flag
    for flag in flags
    if not flag.startswith("--xla_force_host_platform_device_count=")
]
os.environ["XLA_FLAGS"] = " ".join(["--xla_force_host_platform_device_count=4", *kept])

import jax

if jax.local_device_count() < 4:
    raise RuntimeError(
        f"JAX has {jax.local_device_count()} device(s). Restart the kernel so this "
        "cell can set the device count before JAX is imported. Otherwise NumPyro "
        "runs the four chains one after another."
    )

from datetime import UTC, datetime
from pathlib import Path

import arviz as az
import numpyro
from jax.random import PRNGKey
from numpyro import distributions as dist
from numpyro import infer

from vacf.calibration import build_mixing_cases, compile_case, make_log_likelihood
from vacf.parameters import BASE_PARAMS, HOMOGENEOUS_PARAM_BOUNDS, get_nuts_init_values

print(f"devices {jax.local_device_count()}")

## Likelihood

Seven terms, treated as independent: latent infection, notifications, TB deaths, adult pulmonary prevalence, the decline between the two surveys, the high-infectious share of prevalent disease, and the Canberra distance of the mixing matrix. Counts and ratios are normal on the log scale with standard deviation 0.1. Proportions are normal on the logit scale with standard deviation 0.2. The Canberra distance is normal around zero with standard deviation `mixing_dist_sd`, which is itself a prior. A multi-year series is averaged, so it weighs about as much as a single point. A solve that does not succeed contributes `-1e10`.

The solver is the one the summer3 notebook inherited: Dopri5, `rtol = atol = 1e-5`, `dtmax = 1`, `max_steps = 4000`. The spline is folded once per draw. The three mixing scalars are not fixed when the curve is built.


In [ ]:
case = next(item for item in build_mixing_cases() if item.name == "spline")
compiled = compile_case(case)
evaluate = make_log_likelihood(compiled, mixing=case.mixing, fold=case.fold)
print(case.name)


## Sample

Each prior is uniform on the summer3 bounds. The chains start at `get_nuts_init_values()`, the interior point that notebook uses, and share one process. The progress bars are the four chains moving together.


In [ ]:
priors = {name: dist.Uniform(low, high) for name, (low, high) in HOMOGENEOUS_PARAM_BOUNDS.items()}
for name, (low, high) in HOMOGENEOUS_PARAM_BOUNDS.items():
    print(f"{name:28s}  Uniform({low}, {high})")


def model() -> None:
    sampled = {name: numpyro.sample(name, prior) for name, prior in priors.items()}
    params = dict(BASE_PARAMS)
    params.update(sampled)
    numpyro.factor("ll", evaluate(params)["total"])


kernel = infer.NUTS(
    model,
    max_tree_depth=5,
    init_strategy=infer.init_to_value(values=get_nuts_init_values()),
)
mcmc = infer.MCMC(
    kernel,
    num_warmup=100,
    num_samples=100,
    num_chains=4,
    chain_method="parallel",
    progress_bar=True,
)
if mcmc.chain_method != "parallel":
    raise RuntimeError(
        f"Refusing to run with chain_method={mcmc.chain_method!r} on "
        f"{jax.local_device_count()} devices."
    )
print(f"chain_method {mcmc.chain_method}  chains 4  devices {jax.local_device_count()}")
mcmc.run(
    PRNGKey(2),
    extra_fields=("potential_energy", "energy", "num_steps", "accept_prob"),
)

## Posterior

The summer3 notebook writes an ArviZ file under `outputs/local`. `sample_stats.lp` is the joint log posterior of each draw, higher better. This writes the same file, then prints the summary. `outputs/` is gitignored.


In [ ]:
idata = az.from_numpyro(mcmc, log_likelihood=False)
root = Path.cwd()
if not (root / "notebooks").is_dir():
    root = root.parent
out_dir = root / "outputs" / "local"
out_dir.mkdir(parents=True, exist_ok=True)
timestamp = datetime.now(UTC).strftime("%Y%m%dT%H%MZ")
path = out_dir / f"{timestamp}.nc"
idata.to_netcdf(path, engine="h5netcdf")
print(path)
az.summary(idata)
